# XAI Plant Disease Classification — Research Notebook

This notebook runs all 5 experiments from the research paper:
- **Exp 1**: Train ResNet18 → evaluate accuracy
- **Exp 2**: Apply XAI (Grad-CAM, LIME, SHAP) → generate explanations
- **Exp 3**: Deletion test → faithfulness scores
- **Exp 4**: Perturbation test → robustness scores
- **Exp 5**: Comparative analysis → summary table + charts

In [ ]:
# ── Setup ─────────────────────────────────────────────
import sys, os
sys.path.append('..')   # project root

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import yaml, json
from pathlib import Path

# Project modules
from backend.ml.models.resnet_model import PlantDiseaseResNet, load_checkpoint, get_device
from backend.ml.utils.data_utils import build_dataloaders, get_inference_transforms, denormalize
from backend.ml.xai.gradcam       import GradCAM
from backend.ml.xai.lime_explainer import LIMEExplainer
from backend.ml.xai.shap_explainer import SHAPExplainer, build_background_samples
from backend.ml.evaluation.metrics import (
    FaithfulnessEvaluator, RobustnessEvaluator, XAIComparator
)

# Matplotlib style
plt.rcParams.update({
    'figure.facecolor': '#0a0c0f',
    'axes.facecolor':   '#111318',
    'axes.edgecolor':   '#1e2530',
    'axes.labelcolor':  '#8b95a6',
    'text.color':       '#e8ecf2',
    'xtick.color':      '#4d5668',
    'ytick.color':      '#4d5668',
    'grid.color':       '#1e2530',
    'font.family':      'monospace',
})

with open('../configs/config.yaml') as f:
    CONFIG = yaml.safe_load(f)

DEVICE = get_device()
print('Config loaded. Device:', DEVICE)

## Experiment 1: Train Model & Evaluate Accuracy

In [ ]:
# ── Option A: Train from scratch ──────────────────────
# from backend.ml.train import train
# model = train(CONFIG)

# ── Option B: Load existing checkpoint ───────────────
CKPT = '../backend/ml/checkpoints/best_model.pth'

if Path(CKPT).exists():
    model, meta = load_checkpoint(CKPT, DEVICE)
    print(f'Loaded checkpoint. Val acc: {meta["metrics"]["val_acc"]:.4f}')
    with open('../backend/ml/checkpoints/class_mapping.json') as f:
        mapping = json.load(f)
    IDX_TO_CLASS = {int(k): v for k, v in mapping['idx_to_class'].items()}
else:
    print('No checkpoint found. Run train() first.')
    model = PlantDiseaseResNet(num_classes=38, pretrained=True).to(DEVICE)
    IDX_TO_CLASS = {i: f'class_{i}' for i in range(38)}

In [ ]:
# ── Full test-set evaluation ──────────────────────────
from backend.ml.train import full_evaluation

_, _, test_loader, dataset = build_dataloaders(
    data_root   = CONFIG['data']['root'],
    image_size  = CONFIG['data']['image_size'],
    batch_size  = CONFIG['data']['batch_size'],
)

results = full_evaluation(model, test_loader, DEVICE, IDX_TO_CLASS)
print(f'Test Accuracy: {results["accuracy"]:.4f}')
print(results['report'])

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = results['confusion_matrix']
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm, annot=False, fmt='d', cmap='Greens',
    xticklabels=[IDX_TO_CLASS[i] for i in range(len(IDX_TO_CLASS))],
    yticklabels=[IDX_TO_CLASS[i] for i in range(len(IDX_TO_CLASS))],
    ax=ax, linewidths=0.3, linecolor='#0a0c0f'
)
ax.set_title('Confusion Matrix — Test Set', pad=16, fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/evaluation/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 2: Generate XAI Explanations

In [ ]:
# ── Pick test images ──────────────────────────────────
# Use a few images from different classes for comparison
TEST_IMAGES = list(Path(CONFIG['data']['root']).rglob('*.jpg'))[:6]

transform = get_inference_transforms(224)

def load_tensor(img_path):
    pil    = Image.open(img_path).convert('RGB')
    tensor = transform(pil).unsqueeze(0)
    return pil, tensor

In [ ]:
# ── Grad-CAM on all test images ───────────────────────
gcam = GradCAM(model, model.get_last_conv_layer())

gcam_results = []
for img_path in TEST_IMAGES:
    pil, tensor   = load_tensor(img_path)
    hm, pred, conf = gcam.generate(tensor.to(DEVICE))
    original       = denormalize(tensor[0])
    overlay        = GradCAM.overlay_heatmap(original, hm)
    gcam_results.append({'path': img_path, 'original': original, 'overlay': overlay,
                          'heatmap': hm, 'pred': pred, 'conf': conf})
    print(f'{img_path.stem}: pred={IDX_TO_CLASS.get(pred, pred)} conf={conf:.3f}')

gcam.remove_hooks()

In [ ]:
# ── Visualise Grad-CAM grid ───────────────────────────
n = len(gcam_results)
fig, axes = plt.subplots(3, n, figsize=(n * 3, 9))
fig.suptitle('Grad-CAM Explanations', y=1.01, fontsize=14)

for i, r in enumerate(gcam_results):
    axes[0, i].imshow(r['original'])
    axes[0, i].set_title(IDX_TO_CLASS.get(r['pred'], r['pred'])[:20], fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(r['heatmap'], cmap='jet', vmin=0, vmax=1)
    axes[1, i].set_title(f'conf={r["conf"]:.2f}', fontsize=8)
    axes[1, i].axis('off')

    axes[2, i].imshow(r['overlay'])
    axes[2, i].set_title('overlay', fontsize=8)
    axes[2, i].axis('off')

row_labels = ['Original', 'Heatmap', 'Overlay']
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=10)

plt.tight_layout()
Path('../outputs/gradcam').mkdir(parents=True, exist_ok=True)
plt.savefig('../outputs/gradcam/gradcam_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── LIME on first 3 images ────────────────────────────
lime_exp = LIMEExplainer(model, DEVICE, num_samples=500)
lime_results = []

for img_path in TEST_IMAGES[:3]:
    pil, _ = load_tensor(img_path)
    r = lime_exp.explain(pil)
    lime_results.append({'path': img_path, **r})
    print(f'{img_path.stem}: pred={IDX_TO_CLASS.get(r["pred_class"], r["pred_class"])} conf={r["confidence"]:.3f} time={r["computation_time"]:.1f}s')

In [ ]:
# ── SHAP on first 3 images ────────────────────────────
_, train_loader, *_ = build_dataloaders(
    CONFIG['data']['root'], batch_size=CONFIG['data']['batch_size']
)
background = build_background_samples(train_loader, n=50, device=DEVICE)

shap_exp = SHAPExplainer(model, background, DEVICE)
shap_results = []

for img_path in TEST_IMAGES[:3]:
    pil, tensor = load_tensor(img_path)
    r = shap_exp.explain(tensor.to(DEVICE))
    shap_results.append({'path': img_path, **r})
    print(f'{img_path.stem}: pred={IDX_TO_CLASS.get(r["pred_class"], r["pred_class"])} time={r["computation_time"]:.1f}s')

## Experiment 3: Faithfulness Evaluation

In [ ]:
# ── Deletion test on all 3 XAI methods ───────────────
faith_eval = FaithfulnessEvaluator(model, DEVICE)

faith_all = {'GradCAM': [], 'LIME': [], 'SHAP': []}

for i, img_path in enumerate(TEST_IMAGES[:3]):
    pil, tensor = load_tensor(img_path)
    tensor = tensor.to(DEVICE)
    target = int(torch.argmax(model(tensor)).item())

    # Grad-CAM heatmap
    gcam2 = GradCAM(model, model.get_last_conv_layer())
    hm_gc, _, _ = gcam2.generate(tensor)
    gcam2.remove_hooks()
    faith_gc = faith_eval.evaluate(tensor, hm_gc, target)
    faith_all['GradCAM'].append(faith_gc)

    # LIME heatmap
    lr = lime_results[i]
    faith_li = faith_eval.evaluate(tensor, lr['heatmap'], target)
    faith_all['LIME'].append(faith_li)

    # SHAP heatmap
    sr = shap_results[i]
    faith_sh = faith_eval.evaluate(tensor, sr['pos_heatmap'], target)
    faith_all['SHAP'].append(faith_sh)

print('Faithfulness scores (lower AUC = more faithful):')
for method, scores in faith_all.items():
    aucs = [s['auc'] for s in scores]
    print(f'  {method:10s}: mean_AUC={np.mean(aucs):.4f} ± {np.std(aucs):.4f}')

In [ ]:
# ── Plot faithfulness curves ──────────────────────────
COLORS = {'GradCAM': '#f0a832', 'LIME': '#4d9ef5', 'SHAP': '#f06060'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle('Faithfulness — Deletion Test', fontsize=13)

for i in range(3):
    ax = axes[i]
    ax.set_title(TEST_IMAGES[i].stem[:25], fontsize=9)
    for method, scores in faith_all.items():
        s = scores[i]
        ax.plot(s['percentages'], s['confidences'], label=method,
                color=COLORS[method], linewidth=2)
    ax.set_xlabel('Fraction deleted')
    if i == 0:
        ax.set_ylabel('Model confidence')
        ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)
    ax.set_ylim(0, 1)

plt.tight_layout()
Path('../outputs/evaluation').mkdir(parents=True, exist_ok=True)
plt.savefig('../outputs/evaluation/faithfulness_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 4: Robustness Testing

In [ ]:
# ── Robustness under noise for all methods ────────────
robust_eval = RobustnessEvaluator(noise_levels=[0.01, 0.05, 0.10, 0.20], num_repetitions=3)

pil, tensor = load_tensor(TEST_IMAGES[0])
tensor = tensor.to(DEVICE)
target = int(torch.argmax(model(tensor)).item())

# Build explain functions for robustness evaluator
def explain_gradcam(t):
    g = GradCAM(model, model.get_last_conv_layer())
    hm, _, _ = g.generate(t)
    g.remove_hooks()
    return hm

def explain_lime_fn(t):
    pil_img = Image.fromarray(denormalize(t[0].cpu()))
    r = lime_exp.explain(pil_img)
    return r['heatmap']

def explain_shap_fn(t):
    r = shap_exp.explain(t)
    return r['pos_heatmap']

rob_gc = robust_eval.evaluate(tensor, explain_gradcam, 'noise')
rob_li = robust_eval.evaluate(tensor, explain_lime_fn, 'noise')
rob_sh = robust_eval.evaluate(tensor, explain_shap_fn, 'noise')

print('Robustness (SSIM — higher = more stable):')
for name, r in [('GradCAM', rob_gc), ('LIME', rob_li), ('SHAP', rob_sh)]:
    print(f'  {name:10s}: mean_SSIM={r["mean_ssim"]:.4f}  mean_ρ={r["mean_spearman"]:.4f}')

In [ ]:
# ── Plot robustness curves ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Robustness Under Gaussian Noise', fontsize=13)

for ax, metric, label in [
    (axes[0], 'ssim',     'SSIM (higher = stable)'),
    (axes[1], 'spearman', 'Spearman ρ (higher = stable)'),
]:
    for name, r in [('GradCAM', rob_gc), ('LIME', rob_li), ('SHAP', rob_sh)]:
        ax.plot(r['levels'], r[metric], label=name, color=COLORS[name], linewidth=2, marker='o', markersize=5)
    ax.set_xlabel('Noise std')
    ax.set_ylabel(label)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('../outputs/evaluation/robustness_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 5: Final Comparison & Summary

In [ ]:
# ── Full comparison table ─────────────────────────────
comparator = XAIComparator(model, DEVICE)

pil, tensor = load_tensor(TEST_IMAGES[0])
tensor = tensor.to(DEVICE)
target = int(torch.argmax(model(tensor)).item())

comparison = comparator.compare(
    tensor, target,
    methods={
        'GradCAM': explain_gradcam,
        'LIME':    explain_lime_fn,
        'SHAP':    explain_shap_fn,
    }
)

print(XAIComparator.summary_table(comparison))

In [ ]:
# ── Final summary figure ──────────────────────────────
import pandas as pd

df = pd.DataFrame([
    {
        'Method': name,
        'Faithfulness': abs(r['faithfulness_score']),
        'Robustness (SSIM)': r['mean_ssim'],
        'Speed (1/time)': 1 / r['computation_time_s'],
    }
    for name, r in comparison.items()
]).set_index('Method')

# Normalize to [0, 1] for radar comparison
df_norm = (df - df.min()) / (df.max() - df.min() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Final XAI Comparison — Experiment 5', fontsize=13)

# Bar chart
ax = axes[0]
x  = np.arange(len(df.columns))
w  = 0.25
for i, (name, row) in enumerate(df_norm.iterrows()):
    ax.bar(x + i * w, row.values, w, label=name, color=list(COLORS.values())[i], alpha=0.85)
ax.set_xticks(x + w)
ax.set_xticklabels(df.columns, fontsize=9)
ax.set_ylabel('Normalised score')
ax.set_title('Normalised Metric Comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Time bar
ax2 = axes[1]
times = [comparison[m]['computation_time_s'] for m in comparison]
names = list(comparison.keys())
bars  = ax2.bar(names, times, color=[COLORS[m] for m in names], alpha=0.85, width=0.5)
ax2.set_ylabel('Computation time (seconds)')
ax2.set_title('Speed Comparison')
ax2.grid(True, alpha=0.3, axis='y')
for bar, t in zip(bars, times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{t:.2f}s', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/evaluation/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Export results to JSON ────────────────────────────
import json, datetime

export = {
    'timestamp': datetime.datetime.now().isoformat(),
    'model_accuracy': float(results['accuracy']),
    'comparison': {
        name: {
            'faithfulness_auc':   r['faithfulness_auc'],
            'faithfulness_score': r['faithfulness_score'],
            'mean_ssim':          r['mean_ssim'],
            'mean_spearman':      r['mean_spearman'],
            'computation_time_s': r['computation_time_s'],
        }
        for name, r in comparison.items()
    }
}

out_path = '../outputs/evaluation/results_summary.json'
with open(out_path, 'w') as f:
    json.dump(export, f, indent=2)

print(f'Results saved to {out_path}')
print(json.dumps(export, indent=2))